# 第 15 週 實作｜一階線性微分方程與 RK4

可分離變數解決不了 $y'+p(t)y=q(t)$。這週的招式很聰明:乘上一個積分因子,讓左邊變成某個乘積的導數——而那個因子是<strong>推出來的</strong>,不是背的。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜手刻 RK4,和 Euler 比出三個數量級

觀念 6、7 的核心驗證。這格寫出 RK4(<strong>W16、W17 會直接重用</strong>),量它的四階收斂,並和 Euler 正面對比。


In [ ]:
def euler(f, y0, t0, t1, h):
    """顯式尤拉法(W14 Lab2 的同一支)"""
    ts, ys = [t0], [y0]
    while ts[-1] < t1 - 1e-12:
        t, y = ts[-1], ys[-1]
        ys.append(y + h * f(t, y)); ts.append(t + h)
    return np.array(ts), np.array(ys)

def rk4(f, y0, t0, t1, h):
    """四階 Runge-Kutta。W16(梯度流)與 W17(capstone)會重用。"""
    ts, ys = [t0], [y0]
    while ts[-1] < t1 - 1e-12:
        t, y = ts[-1], ys[-1]
        k1 = f(t,       y)
        k2 = f(t + h/2, y + h*k1/2)
        k3 = f(t + h/2, y + h*k2/2)
        k4 = f(t + h,   y + h*k3)
        ys.append(y + h*(k1 + 2*k2 + 2*k3 + k4)/6)
        ts.append(t + h)
    return np.array(ts), np.array(ys)

# 測試:y' = y, y(0)=1 → e^t
f, exact = (lambda t, y: y), math.e

print(f"{'h':>9} {'Euler 誤差':>12} {'比值':>7} {'RK4 誤差':>12} {'比值':>7}")
hs, ee, er = [], [], []
pe = pr = None
for k in range(1, 8):
    h = 0.5**k
    e1 = abs(euler(f, 1.0, 0, 1, h)[1][-1] - exact)
    e2 = abs(rk4(f, 1.0, 0, 1, h)[1][-1] - exact)
    hs.append(h); ee.append(e1); er.append(e2)
    r1 = f"{pe/e1:7.2f}" if pe else "      -"
    r2 = f"{pr/e2:7.2f}" if pr else "      -"
    print(f"{h:9.5f} {e1:12.3e} {r1} {e2:12.3e} {r2}")
    pe, pr = e1, e2

plt.loglog(hs, ee, 'o-', label='Euler')
plt.loglog(hs, er, 's-', label='RK4')
plt.loglog(hs, np.array(hs)*ee[0]/hs[0], 'k:', label='slope 1')
plt.loglog(hs, np.array(hs)**4*er[0]/hs[0]**4, 'k--', label='slope 4')
plt.xlabel('h'); plt.ylabel('|error at t=1|'); plt.legend(fontsize=8)
plt.title('Euler O(h) vs RK4 O(h^4)')
plt.show()

for name, e in [('Euler', ee), ('RK4', er)]:
    print(f"{name:6s} log-log 斜率 = {np.polyfit(np.log10(hs), np.log10(e), 1)[0]:.3f}")

# 成本效益:達到 1e-8 需要多少次函數求值
# 由觀測到的誤差常數 C = E / h^p 外推,不真的跑 —— Euler 會是上億步
print("\n要達到誤差 < 1e-8(由觀測的誤差常數外推,不實跑):")
target = 1e-8
for name, elist, p, per_step in [('Euler', ee, 1, 1), ('RK4', er, 4, 4)]:
    C = elist[-1] / hs[-1]**p          # 用最小的 h 估誤差常數
    h_need = (target / C)**(1/p)
    steps = math.ceil(1/h_need)
    print(f"  {name:6s} 需要 h≈{h_need:.2e}  步數≈{steps:>14,}  "
          f"函數求值≈{steps*per_step:>14,}")
print("  → RK4 每步貴 4 倍,但總求值次數少了好幾個數量級")

In [ ]:
# TODO 學生練習:把 f 換成 lambda t, y: -2*y(精確解 e^(-2t))
# 再試 h = 1.5 的 Euler。它會發散嗎?RK4 呢?(這是觀念 7 的穩定性)

## Lab 2｜混合問題與 RC 電路:同一條方程

觀念 4、5 說兩者數學相同。這格把解析解與數值解對照,並驗證時間常數的 63% 規則。


In [ ]:
from scipy.integrate import solve_ivp

# --- 混合問題:100 L 純水,2 g/L 鹽水以 5 L/min 進出 ---
V, r, cin = 100.0, 5.0, 2.0
mix = lambda t, y: r*cin - r*y[0]/V
sol = solve_ivp(mix, [0, 120], [0.0], dense_output=True, rtol=1e-10, atol=1e-12)

ts = np.linspace(0, 120, 300)
exact_mix = 200*(1 - np.exp(-ts/20))
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(ts, exact_mix, lw=2, label='exact 200(1-e^{-t/20})')
plt.plot(ts, sol.sol(ts)[0], 'r--', lw=1, label='solve_ivp')
plt.axhline(200, color='C2', ls=':', label='final = V*cin = 200 g')
plt.xlabel('t (min)'); plt.ylabel('salt (g)'); plt.legend(fontsize=8)
plt.title('Mixing tank')

print(f"最大誤差 = {np.max(np.abs(sol.sol(ts)[0] - exact_mix)):.2e}")
# 只在積分區間內取值 —— dense_output 對區間外會外插,給出毫無意義的數字
y_end = sol.sol(120)[0]                    # 注意只索引一層
print(f"t=120 min 的鹽量 = {y_end:.4f} g   理論極限 V*cin = {V*cin} g")
print(f"t=120 min 的濃度 = {y_end/V:.4f} g/L  流入濃度 = {cin} g/L  ← 應趨於相等")
print(f"(120 min = 6 個時間常數,已達極限的 {y_end/200*100:.2f}%)")

# --- RC 電路:同一條方程,換個名字 ---
R, C, Vs = 1000.0, 100e-6, 5.0
tau = R*C
rc = lambda t, q: (Vs - q[0]/C)/R
solrc = solve_ivp(rc, [0, 5*tau], [0.0], dense_output=True, rtol=1e-10, atol=1e-14)

t2 = np.linspace(0, 5*tau, 300)
plt.subplot(1, 2, 2)
plt.plot(t2, Vs*(1 - np.exp(-t2/tau)), lw=2, label='exact')
plt.plot(t2, solrc.sol(t2)[0]/C, 'r--', lw=1, label='solve_ivp')
for n in [1, 2, 3, 5]:
    plt.axvline(n*tau, color='0.8', lw=0.8)
plt.axhline(Vs, color='C2', ls=':', label='V = 5')
plt.xlabel('t (s)'); plt.ylabel('capacitor voltage (V)'); plt.legend(fontsize=8)
plt.title(f'RC circuit, tau = {tau} s')
plt.tight_layout(); plt.show()

print(f"\n時間常數 tau = R*C = {tau} s")
print(f"{'t':>8} {'充電比例':>10} {'理論 1-e^-n':>14}")
for n in [1, 2, 3, 5]:
    frac = solrc.sol(n*tau)[0]/(C*Vs)
    print(f"{n}tau {frac:10.4f} {1-math.exp(-n):14.4f}")

In [ ]:
# TODO 學生練習:把混合問題改成「進 3 L/min、出 2 L/min」(體積會變!)
# V(t) = 50 + t,方程變成 y' = 3*cin - 2*y/(50+t)
# 這時 p(t) 不是常數 —— 用 solve_ivp 解,並和積分因子法的解析解對照

## Lab 3｜穩定性:步長太大會發散

觀念 7 說「階數管精度,穩定性管會不會炸」。這格用 $y'=-\lambda y$ 演示:同一個方法,$h$ 過了門檻就完全爆掉。


In [ ]:
def euler(f, y0, t0, t1, h):
    ts, ys = [t0], [y0]
    while ts[-1] < t1 - 1e-12:
        t, y = ts[-1], ys[-1]
        ys.append(y + h*f(t, y)); ts.append(t + h)
    return np.array(ts), np.array(ys)

lam = 10.0                                # y' = -10 y
f = lambda t, y: -lam*y
h_crit = 2/lam                            # Euler 的穩定門檻

print(f"y' = -{lam}y 的 Euler 穩定條件:|1 + h*(-{lam})| < 1  →  h < {h_crit}")
print(f"\n{'h':>7} {'|1+h*lam_neg|':>16} {'最終 |y|':>14} {'狀態'}")
for h in [0.05, 0.1, 0.15, 0.19, 0.21, 0.25, 0.3]:
    amp = abs(1 - h*lam)
    ts, ys = euler(f, 1.0, 0, 2, h)
    final = abs(ys[-1])
    state = "穩定" if amp < 1 else "發散"
    print(f"{h:7.2f} {amp:16.3f} {final:14.4e}   {state}"
          f"{'  ← 剛好在門檻上' if abs(h-h_crit) < 1e-9 else ''}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
tt = np.linspace(0, 2, 300)
for h, style in [(0.05, 'C0'), (0.15, 'C1'), (0.21, 'C3'), (0.25, 'C4')]:
    ts, ys = euler(f, 1.0, 0, 2, h)
    which = 0 if abs(1-h*lam) < 1 else 1
    ax[which].plot(ts, ys, 'o-', ms=3, color=style, label=f'h={h}')
for a, title in zip(ax, ['Stable: h < 0.2', 'Unstable: h > 0.2  (note the y-scale!)']):
    a.plot(tt, np.exp(-lam*tt), 'k--', lw=1, label='exact')
    a.set_title(title); a.legend(fontsize=8); a.set_xlabel('t')
plt.tight_layout(); plt.show()

print("\n→ h = 0.21 只比門檻大 5%,解卻整個震盪爆炸。")
print("  這不是『不夠準』,是『完全錯』—— 而且不會有任何錯誤訊息。")
print("  W16 會看到:learning rate 太大導致訓練發散,是同一件事。")

In [ ]:
# TODO 學生練習:把 lam 改成 100。穩定門檻變成多少?
# 再試 RK4(用 Lab1 的函式)——它的穩定門檻比 Euler 大還是小?